## Step 0: Install Libraries

In [ ]:
!pip install geopandas osmnx tensorflow matplotlib numpy contextily opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 87.2 MB/s eta 0:00:00


## Step 1: Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import osmnx as ox
import contextily as ctx
from tensorflow.keras import layers
from pathlib import Path

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

policy = tf.keras.mixed_precision.Policy('mixed_float16')
tf.keras.mixed_precision.set_global_policy(policy)

## Step 2: Loading Dataset

In [ ]:
path = "/content/drive/MyDrive/GANs-Based-City-Layout/Gurugram/map_tiles2"
def create_dataset_from_drive(drive_folder_path=path, tile_size=256, batch_size=16, num_samples=2500):
    if not os.path.exists(drive_folder_path):
        print(f"Error: {drive_folder_path} does not exist. Please check the path.")
        return None

    !ls -l "{drive_folder_path}" | head -n 10
    print(f"Accessing tiles from {drive_folder_path}")

    def load_image(file_path):
        img = tf.io.read_file(file_path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.2)
        img = img * 2.0 - 1.0
        return img

    dataset = tf.data.Dataset.list_files(f"{drive_folder_path}/*.png")
    dataset = (dataset
               .shuffle(5000)
               .take(num_samples)
               .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
               .batch(batch_size)
               .prefetch(tf.data.AUTOTUNE))
    return dataset

In [ ]:
dataset = create_dataset_from_drive(drive_folder_path=path, num_samples=3000)
if dataset:
    print("Dataset created from California tiles with augmentation and prefetching.")
else:
    print("Dataset creation failed. Please verify the Drive folder path.")

total 491466
-rw------- 1 root root 115079 Apr 16 06:05 tile_0.png
-rw------- 1 root root  44313 Apr 16 06:05 tile_1000.png
-rw------- 1 root root  46689 Apr 16 06:05 tile_1001.png
-rw------- 1 root root  31355 Apr 16 06:05 tile_1002.png
-rw------- 1 root root  31355 Apr 16 06:05 tile_1003.png
-rw------- 1 root root  31355 Apr 16 06:05 tile_1004.png
-rw------- 1 root root  56064 Apr 16 06:05 tile_1005.png
-rw------- 1 root root  61763 Apr 16 06:05 tile_1006.png
-rw------- 1 root root  61763 Apr 16 06:05 tile_1007.png
Accessing tiles from /content/drive/MyDrive/GANs-Based-City-Layout/Gurugram/map_tiles2
Dataset created from California tiles with augmentation and prefetching.


## Step 3: MODEL

In [ ]:
def make_generator(latent_dim=128):
    model = tf.keras.Sequential([
        layers.Input(shape=(latent_dim,)),
        layers.Dense(4 * 4 * 1024),
        layers.LeakyReLU(0.2),
        layers.Reshape((4, 4, 1024)),
        layers.Conv2DTranspose(512, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(256, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(128, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(64, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(32, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(3, (4, 4), strides=(2, 2), padding='same', activation='tanh')
    ])
    return model

In [ ]:
def make_discriminator(img_size=256):
    model = tf.keras.Sequential([
        layers.Input(shape=(img_size, img_size, 3)),
        layers.Conv2D(64, (4, 4), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Conv2D(128, (4, 4), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Conv2D(256, (4, 4), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Conv2D(512, (4, 4), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

In [ ]:
latent_dim = 128
generator = make_generator(latent_dim)
discriminator = make_discriminator()

## Step 4: Loss Function

In [ ]:
def wasserstein_loss(y_true, y_pred):
    return tf.reduce_mean(y_true * y_pred)

In [ ]:
def gradient_penalty(real_images, fake_images, discriminator):
    real_images = tf.cast(real_images, tf.float16)
    fake_images = tf.cast(fake_images, tf.float16)
    alpha = tf.cast(tf.random.uniform([real_images.shape[0], 1, 1, 1], 0., 1.), tf.float16)

    interpolates = alpha * real_images + (1 - alpha) * fake_images
    with tf.GradientTape() as gp_tape:
        gp_tape.watch(interpolates)
        pred = discriminator(interpolates, training=True)
    grads = gp_tape.gradient(pred, [interpolates])[0]
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]))
    return tf.reduce_mean((norm - 1.) ** 2)

## Step 5: Compilation

In [ ]:
gen_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5, beta_2=0.9)
disc_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5, beta_2=0.9)
gp_weight = tf.cast(1.0, tf.float16)

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], latent_dim])
    with tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        disc_loss_real = wasserstein_loss(tf.ones_like(real_output), real_output)
        disc_loss_fake = wasserstein_loss(-tf.ones_like(fake_output), fake_output)
        gp = gradient_penalty(images, generated_images, discriminator)
        disc_loss = disc_loss_real + disc_loss_fake + gp_weight * gp

    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    with tf.GradientTape() as gen_tape:
        generated_images = generator(noise, training=True)
        fake_output = discriminator(generated_images, training=True)
        gen_loss = wasserstein_loss(tf.ones_like(fake_output), fake_output)

    gen_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gen_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
checkpoint_dir = './checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator=generator, discriminator=discriminator,
                                 gen_optimizer=gen_optimizer, disc_optimizer=disc_optimizer)

## Step 6: Generation

In [ ]:
def generate_and_save_images(model, epoch, save_folder='/content/drive/MyDrive/GANs-Based-City-Layout/Gurugram/generated_images2'):
    noise = tf.random.normal([16, latent_dim])
    predictions = model(noise, training=False)
    predictions = (predictions + 1) / 2.0
    predictions = tf.cast(predictions, tf.float32)
    Path(save_folder).mkdir(parents=True, exist_ok=True)
    fig, axs = plt.subplots(4, 4, figsize=(12, 12))
    for i, ax in enumerate(axs.flat):
        ax.imshow(predictions[i])
        ax.axis('off')
    plt.savefig(os.path.join(save_folder, f"epoch_{epoch}.png"))
    plt.close()

In [ ]:
def save_generator_model(model, filepath='./generator_model.keras', zip_filename='generator_model.zip'):
    model.save(filepath, save_format='keras_v3')
    !zip -r {zip_filename} {filepath}
    from google.colab import files
    files.download(zip_filename)
    print(f"Generator model saved and downloaded as '{zip_filename}'")

In [ ]:
def train(dataset, epochs=50, disc_steps=1):
    for epoch in range(epochs):
        for image_batch in dataset:
            for _ in range(disc_steps):
                gen_loss, disc_loss = train_step(image_batch)

        if (epoch + 1) % 5 == 0:
            generate_and_save_images(generator, epoch + 1)
            checkpoint.save(file_prefix=checkpoint_prefix)
            print(f"Epoch {epoch + 1}, Gen Loss: {gen_loss:.4f}, Disc Loss: {disc_loss:.4f}")

    # save_generator_model(generator)

In [ ]:
if dataset:
    train(dataset, epochs=100)
    print("Training complete. High-quality city layout images saved in '/content/drive/My Drive/GANs-Urban-Layout/Gurugram/generated_images2'.")
else:
    print("Training creation failed. Please verify the Drive folder path.")

Epoch 5, Gen Loss: -3898.0000, Disc Loss: -459.5000
Epoch 10, Gen Loss: -488.0000, Disc Loss: -149.0000
Epoch 15, Gen Loss: 175.2500, Disc Loss: 18.4688
Epoch 20, Gen Loss: -12.7500, Disc Loss: -16.7344
Epoch 25, Gen Loss: -849.0000, Disc Loss: -221.3750
Epoch 30, Gen Loss: -896.5000, Disc Loss: -298.5000
Epoch 35, Gen Loss: -1137.0000, Disc Loss: -327.0000
Epoch 40, Gen Loss: -1930.0000, Disc Loss: -555.0000
Epoch 45, Gen Loss: -2832.0000, Disc Loss: -193.7500
Epoch 50, Gen Loss: -1861.0000, Disc Loss: 72.2500
Epoch 55, Gen Loss: -2226.0000, Disc Loss: -24.1250
Epoch 60, Gen Loss: -1554.0000, Disc Loss: -350.5000
Epoch 65, Gen Loss: -2604.0000, Disc Loss: 937.0000
Epoch 70, Gen Loss: -2210.0000, Disc Loss: -837.5000
Epoch 75, Gen Loss: -1919.0000, Disc Loss: -436.5000
Epoch 80, Gen Loss: -2174.0000, Disc Loss: -886.0000
Epoch 85, Gen Loss: -2174.0000, Disc Loss: -187.2500
Epoch 90, Gen Loss: -2004.0000, Disc Loss: -580.0000
Epoch 95, Gen Loss: -2070.0000, Disc Loss: 51.0000
Epoch 100,

## New Layout Generation

In [ ]:
save_generator_model(generator)

  adding: generator_model.keras (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Generator model saved and downloaded as 'generator_model.zip'


In [ ]:
model_path="/content/drive/MyDrive/GANs-Based-City-Layout/California/generatedModel/generator_model.keras"
def generate_new_images(model_path="/content/generator_model.keras", num_images=50, save_folder='/content/drive/MyDrive/GANs-Based-City-Layout/Gurugram/new_images2'):
    loaded_generator = tf.keras.models.load_model(model_path)
    noise = tf.random.normal([num_images, latent_dim])
    predictions = loaded_generator(noise, training=False)
    predictions = (predictions + 1) / 2.0
    predictions = tf.cast(predictions, tf.float32)

    if len(predictions.shape) == 2 and predictions.shape[1] == 196608:  # 256*256*3
        predictions = tf.reshape(predictions, [num_images, 256, 256, 3])
    elif predictions.shape[1:] != [256, 256, 3]:
        raise ValueError(f"Unexpected prediction shape: {predictions.shape}. Expected [num_images, 256, 256, 3]")

    Path(save_folder).mkdir(parents=True, exist_ok=True)
    for i in range(num_images):
        plt.figure(figsize=(5, 5))
        plt.imshow(predictions[i])
        plt.axis('off')
        plt.savefig(os.path.join(save_folder, f"brand_new_world_{i}.png"))
        plt.close()
    print(f"{num_images} new images generated and saved to {save_folder}")

In [ ]:
generate_new_images()